### CRISP-DM Phase 3 - Data Preparation : Earth Observation

Cleaning and preprocessing of selected datasets from **Climate Data Store** by *Copernicus Climate Change Service*

In [ ]:
import regionmask
import pandas as pd
import zipfile
import glob
import xarray as xr
import cfgrib
from iso3166 import countries_by_alpha2

DP1 - Collecting

In [ ]:
countries = regionmask.defined_regions.natural_earth_v5_0_0.countries_110

def grid_to_countries(df, var_name, lat=None, lon=None):
    if lat is None:
        lat = 'latitude' if 'latitude' in df.coords else 'lat'
    if lon is None:
        lon = 'longitude' if 'longitude' in df.coords else 'lon'
        
    mask = countries.mask(df[lon], df[lat])
    records = []
    for i, country in enumerate(countries):
        country_mask = mask == i
        var_country = df[var_name].where(country_mask)
        var_mean = var_country.mean(dim=[lat, lon])
        df_country = var_mean.to_dataframe(name='value').reset_index()
        df_country['country'] = country.name
        df_country['iso'] = country.abbrev
        df_country['variable'] = var_name
        records.append(df_country)
    return pd.concat(records, ignore_index=True)

In [ ]:
# Unzip each folder, change with associated name
!unzip -q "2m_temperature.zip" -d "data/2m_temperature"

In [ ]:
# .zip folders containing .grib files
grib_files = {'2m_temperature': '2m_temperature.zip', 'instantaneous_wind_gust': 'instantaneous_wind_gust.zip',
    'snowmelt': 'snowmelt.zip', 'total_precipitation': 'total_precipitation.zip'}

for name, zip_file in grib_files.items():  
    # Load .grib file
    grib_path = glob.glob(f'data/{name}/*.grib')[0]
    ds = xr.open_dataset(grib_path, engine='cfgrib')
    print(f"Variables: {list(ds.data_vars)}")
    print(f"Dimensions: {dict(ds.dims)}")
    
    # Ask user which variable to use from the dataset
    if len(ds.data_vars) > 1:
        print(f'Select the variable to use for {name}')
        for i, var in enumerate(ds.data_vars):
            print(f"{i}: {var}")
        var_index = int(input("Enter the number of the variable to use: "))
        var = list(ds.data_vars)[var_index]
    else:
        var = list(ds.data_vars)[0]
    
    # Aggregate
    df = grid_to_countries(ds, var, lat='latitude', lon='longitude')
    df.to_csv(f'data/{name}.csv', index=False)

In [ ]:
# .zip folders containing .nc files 
nc_files = {'sea_level_anomaly': 'sea_level_anomaly.zip', 'spei': 'spei.zip'}

for name, zip_file in nc_files.items():  
    # Extract .zip file
    z = zipfile.ZipFile(f'data/{zip_file}')
    z.extractall(f'data/{name}')
    
    # Load .nc files
    files = files = sorted(glob.glob(f'data/{name}/*.nc'))
    ds = xr.open_mfdataset(files, combine='by_coords')
    
    # Ask user which variable to use from the dataset
    if len(ds.data_vars) > 1:
        print(f'Select the variable to use for {name}')
        for i, var in enumerate(ds.data_vars):
            print(f"{i}: {var}")
        var_index = int(input("Enter the number of the variable to use: "))
        var = list(ds.data_vars)[var_index]
    else:
        var = list(ds.data_vars)[0]
    
    # Aggregate
    df = grid_to_countries(ds, var)
    df.to_csv(f'data/{name}.csv', index=False)

DP2 - Harmonizing

In [ ]:
# Load the datasets
temperature = pd.read_csv('data/2m_temperature.csv')
wind = pd.read_csv('data/instantaneous_wind_gust.csv')
sea = pd.read_csv('data/sea_level_anomaly.csv')
snowmelt = pd.read_csv('data/snowmelt.csv')
spei = pd.read_csv('data/spei.csv')
precipitation = pd.read_csv('data/total_precipitation.csv')

In [ ]:
## Select and rename columns
temperature = temperature[['iso', 'country', 'valid_time', 'value']].copy()
temperature.columns = ['Country', 'Country_name', 'Date', '2m_temperature']

wind = wind[['iso', 'country', 'valid_time', 'value']].copy()
wind.columns = ['Country', 'Country_name', 'Date', 'Instantaneous_wind_gust']

sea = sea[['iso', 'country', 'time', 'value']].copy()
sea.columns = ['Country', 'Country_name', 'Date', 'Sea_level_anomaly']

snowmelt = snowmelt[['iso', 'country', 'valid_time', 'value']].copy()
snowmelt.columns = ['Country', 'Country_name', 'Date', 'Snowmelt']

spei = spei[['iso', 'country', 'time', 'value']].copy()
spei.columns = ['Country', 'Country_name', 'Date', 'SPEI']

precipitation = precipitation[['iso', 'country', 'valid_time', 'value']].copy()
precipitation.columns = ['Country', 'Country_name', 'Date', 'Total_precipitation']

In [ ]:
var_dict = {'2m temperature': temperature, 'Instantaneous wind gust': wind, 
            'Sea level anomaly': sea, 'Snowmelt': snowmelt, 
            'SPEI': spei, 'Total precipitation': precipitation}

DP3 - Cleaning

In [ ]:
## Clean ISO codes
# Change ISO alpha 2 codes to alpha 3 codes
def change_iso(iso):
    code = str(iso).strip()
    if len(code) == 3:
        return code
    try:
        result = countries_by_alpha2.get(iso)
        return result.alpha3 if result else iso
    except:
        return iso
        
# Manual fixes from noticed issues
iso_fixes = {'INDO': 'IDN', 'DRC': 'COD', 'RUS': 'RUS', 'N': 'NOR', 'F': 'FRA', 'PAL': 'PSE',
             'IRQ': 'IRQ', 'IND': 'IND', 'IRN': 'IRN', 'SYR': 'SYR', 'ARM': 'ARM', 'S': 'SWE', 
             'A': 'AUT', 'EST': 'EST', 'D': 'DEU', 'L': 'LUX', 'B': 'BEL', 'P': 'PRT', 'E': 'ESP', 
             'IRL': 'IRL', 'I': 'ITA', 'SLO': 'SVN', 'FIN': 'FIN', 'BiH': 'BIH', 'NM': 'MKD', 
             'KO': 'XKX', 'ES': 'SWZ'}
country_fixes = {'Jamaica': 'JAM', 'Jordan': 'JOR', 'Japan': 'JPN', 'Namibia': 'NAM'}

def clean_iso(df):
    df['Country'] = df['Country'].replace(iso_fixes)
    df['Country'] = df.apply(lambda row: country_fixes.get(row['Country_name'], row['Country']), axis=1)
    df['Country'] = df['Country'].apply(change_iso)   
    return df

In [ ]:
## Clean date
def clean_date(df):
    df['Date'] = pd.to_datetime(df['Date']).dt.to_period('M').dt.to_timestamp()
    return df

In [ ]:
## Apply cleaning steps
for name, df in var_dict.items():
    df = clean_iso(df)
    df = clean_date(df)

DP4 - Merging

In [ ]:
sensor = var_dict['2m temperature'].copy()
for name, df in var_dict.items():
    if name == '2m temperature':
        continue
    sensor = df.merge(sensor, on=['Country', 'Country_name', 'Date'], how='outer')

In [ ]:
## Add a 'Year' column to match with law data
sensor['Year'] = pd.to_datetime(sensor['Date']).dt.year

In [ ]:
## Rearrange columns
sensor = sensor[['Country', 'Country_name', 'Year', 'Date', '2m_temperature', 'Instantaneous_wind_gust',
                 'Sea_level_anomaly', 'Snowmelt', 'SPEI', 'Total_precipitation']].copy()

In [ ]:
## Export .csv
sensor.to_csv('data/sensor.csv', index=False)